# T03 — Feature Engineering
**Input:** `data/loaded/train_rul.csv`, `data/loaded/test_rul.csv`  
**Goal:** drop constant sensors → normalize per (dataset_id, op_cluster) → add rolling features  
**Output:** `data/processed/train_features.csv`, `data/processed/test_features.csv`  
**Artifacts:** `artifacts/kmeans_op_clusters.pkl`, `artifacts/scalers.pkl`  
**Next:** modeling notebooks (ARIMA, LSTM, TFT, ...)

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import pandas as pd
from src.preprocessing.cleaning import drop_sensors, get_sensor_cols
from src.preprocessing.scaling import (
    scale_sensors,
    verify_scaling,
    save_scaling_artifacts,
)
from src.features.rolling import add_rolling_features, verify_rolling_features

LOADED_DIR      = ROOT / "data" / "loaded"
PROC_DIR     = ROOT / "data" / "processed"
ARTIFACTS_DIR = ROOT / "artifacts"
PROC_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

## Load from T02 outputs

In [ ]:
train = pd.read_csv(LOADED_DIR / "train_rul.csv")
test  = pd.read_csv(LOADED_DIR / "test_rul.csv")
print(f"train: {train.shape}  |  test: {test.shape}")

---
## Step 1 — Drop low-variance sensors
detected on train only (never test) — per-dataset_id variance check  
a sensor must be flat in ALL subsets to be dropped (not just FD001)

In [ ]:
train, test, dropped_sensors = drop_sensors(train, test)
sensor_cols = get_sensor_cols(train)
print(f"\nSensors kept ({len(sensor_cols)}): {sensor_cols}")

---
## Step 2 — Normalize per (dataset_id, op_cluster) using StandardScaler
StandardScaler chosen over MinMaxScaler: test values can exceed train extremes  
as engines degrade — StandardScaler handles this as high z-scores, not out-of-range values  
KMeans n_clusters adapts automatically for FD001/FD003 (single operating condition)  
fitted artifacts saved to `artifacts/` for inference reuse

In [ ]:
train, test, km, scalers = scale_sensors(train, test, sensor_cols)
verify_scaling(train, sensor_cols)

# persist so downstream inference / other notebooks don't need to refit
save_scaling_artifacts(km, scalers, ARTIFACTS_DIR)

In [ ]:
# op_cluster distribution — FD001/FD003 should use fewer clusters than FD002/FD004
assert "op_cluster" in train.columns and "op_cluster" in test.columns
print("op_cluster distribution (train):")
print(train.groupby("dataset_id")["op_cluster"].value_counts().sort_index().to_string())

---
## Step 3 — Rolling features
input is sorted by (engine_id, cycle) inside add_rolling_features  
rolling windows never cross engine boundaries

In [ ]:
ROLLING_WINDOWS = [5, 10, 20]  # 20 added: covers full LSTM window at every position

train = add_rolling_features(train, sensor_cols, windows=ROLLING_WINDOWS)
test  = add_rolling_features(test,  sensor_cols, windows=ROLLING_WINDOWS)

n_rolling = len(sensor_cols) * len(ROLLING_WINDOWS) * 2
print(f"Added {n_rolling} rolling feature columns")
print(f"train: {train.shape}  |  test: {test.shape}")

## Verification

In [ ]:
# no NaN anywhere
assert train.isnull().sum().sum() == 0, "NaN in train"
assert test.isnull().sum().sum()  == 0, "NaN in test"
print("[PASS] no NaN values")

# dropped sensors are gone
for s in dropped_sensors:
    assert s not in train.columns, f"{s} still in train"
print(f"[PASS] all {len(dropped_sensors)} dropped sensors absent")

# rul_last must not be present (dropped in T02, verified again here for safety)
assert "rul_last" not in train.columns and "rul_last" not in test.columns
print("[PASS] rul_last absent — no leakage risk")

# RUL values must be unchanged from T02 input
train_rul_check = pd.read_csv(LOADED_DIR / "train_rul.csv")["RUL"]
# T03 re-sorts by (engine_id, cycle) internally — align on engine_id+cycle not position
ref = pd.read_csv(LOADED_DIR / "train_rul.csv")[["engine_id", "cycle", "RUL"]]
check = train[["engine_id", "cycle", "RUL"]]
merged = check.merge(ref, on=["engine_id", "cycle"], suffixes=("_new", "_ref"))
assert (merged["RUL_new"] == merged["RUL_ref"]).all(), "RUL values changed during feature engineering!"
print("[PASS] RUL values unchanged")

# rolling feature integrity
verify_rolling_features(train, sensor_cols, ROLLING_WINDOWS)

## Save

In [ ]:
train.to_csv(PROC_DIR / "train_features.csv", index=False)
test.to_csv(PROC_DIR / "test_features.csv", index=False)
print(f"Saved: {PROC_DIR / 'train_features.csv'}")
print(f"Saved: {PROC_DIR / 'test_features.csv'}")

## Feature summary for modeling notebooks

In [ ]:
rolling_cols = [c for c in train.columns if "_rmean_" in c or "_rstd_" in c]
all_feature_cols = sensor_cols + rolling_cols

print(f"Raw sensor features:    {len(sensor_cols)}")
print(f"Rolling features:       {len(rolling_cols)}")
print(f"Total feature columns:  {len(all_feature_cols)}")
print(f"\nNon-feature columns kept: engine_id, cycle, op1, op2, op3, dataset_id, op_cluster, RUL")
print(f"\nAll feature cols for X: {all_feature_cols}")